In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [2]:
df = pd.read_parquet("/Users/jasonc/Desktop/DSC_291/merged_features.parquet")

In [3]:
df

,total_amount,mean_amount,std_amount,min_amount,max_amount,median_amount,transaction_count,credit_sum,debit_sum,credit_debit_ratio,...,fft_high_freq_power_cat31.0,fft_power_ratio_cat31.0,fft_dominant_freq_cat33.0,fft_dominant_power_cat33.0,fft_spectral_entropy_cat33.0,fft_low_freq_power_cat33.0,fft_high_freq_power_cat33.0,fft_power_ratio_cat33.0,dataset_type,FPF_TARGET
masked_consumer_id,,,,,,,,,,,,,,,,,,,,,
C01100001,-8111.42,-3.051701,645.540740,-8000.00,8483.00,-25.380,2658,276864.26,284975.68,0.971536,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100002,-9490.73,-3.813070,910.144154,-19150.00,20000.00,-25.290,2489,285876.77,295367.50,0.967868,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100003,2137.77,3.224389,748.056684,-4310.28,4409.40,-21.350,663,84178.75,82040.98,1.026057,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100004,-29837.87,-27.525710,414.375001,-1800.00,1748.00,-27.555,1084,97426.62,127264.49,0.765544,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100005,-9756.37,-4.044930,1314.772339,-6987.07,39915.16,-25.080,2412,404922.85,414679.22,0.976472,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C01104996,38478.08,12.384319,1634.482084,-37648.00,44100.00,-26.530,3107,470767.11,432289.03,1.089010,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104997,591.59,0.512199,4459.475583,-56256.44,134384.01,-19.990,1155,217927.74,217336.15,1.002722,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104998,-12120.38,-3.429649,394.044758,-1953.37,12710.00,-17.715,3534,136083.82,148204.20,0.918218,...,0.0,0.0,0.210526,16857.338694,1.894851,0.0,21227.966021,0.0,real,0.0


In [4]:
# Group by customer_category and FPF_TARGET, then count occurrences
category_target_counts = (
    df.groupby(["dataset_type", "FPF_TARGET"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: "count_0", 1.0: "count_1"})
)
category_target_counts

FPF_TARGET,count_0,count_1
dataset_type,,
real,3942,58


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.utils import resample
from xgboost import XGBClassifier

# ========== Load and Prepare Data ==========

df = df.reset_index().rename(columns={'index': 'masked_consumer_id'})
df['masked_consumer_id'] = df['masked_consumer_id'].astype(str)

drop_cols = ['masked_consumer_id', 'FPF_TARGET', 'dataset_type']
X = df.drop(columns=drop_cols)
y = df['FPF_TARGET']
feat_names = X.columns.tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ========== Feature Selection with RFECV ==========

print("→ Performing RFECV with logistic regression...")

estimator = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    max_iter=2000,
    random_state=42
)

rfecv = RFECV(
    estimator=estimator,
    step=50,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    min_features_to_select=175
)

rfecv.fit(X_train, y_train)
support_mask = rfecv.support_
selected_feats = [f for f, keep in zip(feat_names, support_mask) if keep]

print(f"Selected {len(selected_feats)} features using RFECV.")

X_train_sel = X_train[selected_feats].reset_index(drop=False)  # retain index
y_train = y_train.reset_index(drop=True)
X_val_sel = X_val[selected_feats]

# ========== Define Final Model ==========

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=20,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.7,
        scale_pos_weight=1,
        eval_metric='logloss',
        random_state=42
    ))
])

# ========== Stratified Bootstrapping ==========

def stratified_bootstrap(X, y, random_state=None):
    """Bootstrap sample with class balance preserved."""
    X_pos = X[y == 1]
    y_pos = y[y == 1]
    X_neg = X[y == 0]
    y_neg = y[y == 0]

    X_pos_bs, y_pos_bs = resample(X_pos, y_pos, replace=True, random_state=random_state)
    X_neg_bs, y_neg_bs = resample(X_neg, y_neg, replace=True, random_state=random_state)

    X_bs = pd.concat([X_pos_bs, X_neg_bs]).reset_index(drop=True)
    y_bs = pd.concat([y_pos_bs, y_neg_bs]).reset_index(drop=True)

    return X_bs, y_bs

print("→ Running stratified bootstrapping (50 rounds)...")

n_bootstraps = 50
rng = np.random.RandomState(42)
bootstrap_aucs = []

for i in range(n_bootstraps):
    X_bs, y_bs = stratified_bootstrap(X_train_sel, y_train, random_state=rng.randint(0, 1_000_000))

    # Identify OOB samples based on original index
    in_bs_orig_idx = set(X_bs['index'])
    oob_mask = ~X_train_sel['index'].isin(in_bs_orig_idx)
    if oob_mask.sum() == 0:
        continue  # skip if no OOB

    X_oob = X_train_sel[oob_mask].drop(columns=['index'])
    y_oob = y_train[oob_mask]

    X_bs = X_bs.drop(columns=['index'])

    model = clone(pipeline)
    model.fit(X_bs, y_bs)
    p_oob = model.predict_proba(X_oob)[:, 1]
    auc = roc_auc_score(y_oob, p_oob)
    bootstrap_aucs.append(auc)
    print(f"   Bootstrap {i+1:2d}: OOB AUC = {auc:.3f}")

# Summarize bootstrapped results
if bootstrap_aucs:
    mean_auc = np.mean(bootstrap_aucs)
    std_auc = np.std(bootstrap_aucs)
    ci_lower = np.percentile(bootstrap_aucs, 2.5)
    ci_upper = np.percentile(bootstrap_aucs, 97.5)

    print(f"→ Stratified Bootstrapped AUC: mean = {mean_auc:.3f}, std = {std_auc:.3f}, 95% CI = [{ci_lower:.3f}, {ci_upper:.3f}]")
else:
    print("⚠️ No valid bootstrap rounds completed (OOB sets may have been empty).")

# ========== Final Model Evaluation on Holdout ==========

print("→ Training final XGBoost model on full training set...")
pipeline.fit(X_train_sel.drop(columns=['index']), y_train)

p_train = pipeline.predict_proba(X_train_sel.drop(columns=['index']))[:, 1]
p_val   = pipeline.predict_proba(X_val_sel)[:, 1]

train_auc = roc_auc_score(y_train, p_train)
val_auc   = roc_auc_score(y_val, p_val)

print(f"   Train AUC = {train_auc:.3f}, Val AUC = {val_auc:.3f}")

→ Performing RFECV with logistic regression...
Selected 175 features using RFECV.
→ Running stratified bootstrapping (50 rounds)...
   Bootstrap  1: OOB AUC = 0.802
   Bootstrap  2: OOB AUC = 0.715
   Bootstrap  3: OOB AUC = 0.637
   Bootstrap  4: OOB AUC = 0.679
   Bootstrap  5: OOB AUC = 0.773
   Bootstrap  6: OOB AUC = 0.750
   Bootstrap  7: OOB AUC = 0.762
   Bootstrap  8: OOB AUC = 0.629
   Bootstrap  9: OOB AUC = 0.632
   Bootstrap 10: OOB AUC = 0.830
   Bootstrap 11: OOB AUC = 0.748
   Bootstrap 12: OOB AUC = 0.738
   Bootstrap 13: OOB AUC = 0.775
   Bootstrap 14: OOB AUC = 0.653
   Bootstrap 15: OOB AUC = 0.729
   Bootstrap 16: OOB AUC = 0.715
   Bootstrap 17: OOB AUC = 0.732
   Bootstrap 18: OOB AUC = 0.772
   Bootstrap 19: OOB AUC = 0.620
   Bootstrap 20: OOB AUC = 0.732
   Bootstrap 21: OOB AUC = 0.777
   Bootstrap 22: OOB AUC = 0.797
   Bootstrap 23: OOB AUC = 0.669
   Bootstrap 24: OOB AUC = 0.730
   Bootstrap 25: OOB AUC = 0.750
   Bootstrap 26: OOB AUC = 0.709
   Bootstr